# Дообучение: веса прошлого прогона -> новый датасет

Сеть стартует **с весов готового прогона** (`INIT_RUN`, по умолчанию `v4_ft`) и учится на новом датасете
(`DATASET_ZIP`, по умолчанию **`dataset_v5`**). Что нового в v5 (разметка `labels-4`):
* **нет точек - нет помещения**: неотснятое (помещение без стоянки, почти невидимое) - улица, а не ignore;
  сеть перестаёт дорисовывать помещения в пустоте у края съёмки;
* **открытые балконы** (без потолка, пол ниже на 12-25 см, ограждение) - пока улица, на плане их нет;
* **верхний этаж двухуровневой квартиры** - пол найден свой, а не потолок нижнего этажа под ним;
* стоянки в каждом помещении, кроме изредка коридора.

**Почему новый прогон, а не `--resume` старого.** Датасет другой, поэтому оптимизатор и расписание LR - заново
(одноцикловое, пик `LR`), а веса - из `INIT_RUN`. Пик LR ниже, чем при обучении с нуля (`3e-4` против `1e-3`):
веса уже хорошие, их надо подправить, а не сломать. Прогон пишется в `plannet/runs/<RUN_NAME>`; `INIT_RUN` не трогается.

**Перед запуском**
1. *Среда выполнения → Сменить тип среды → GPU*.
2. На Google Drive в `plannet`: архив датасета (`python -m simscan dataset pack ... --out dataset_v5.zip`) и прогон
   `runs/<INIT_RUN>/` с весами. Берутся `best.pt` (лучший по плану); если его нет - `best_pixel.pt`, затем `last.pt`.
3. Секрет `GITHUB_TOKEN` - как в основном ноутбуке.
4. *Среда выполнения → Выполнить все.*

**Следующий датасет** - поменять `DATASET_ZIP`, `INIT_RUN` (обычно - прошлый `RUN_NAME`) и `RUN_NAME`.

**`v5_ft` не продолжать.** В нём стёртое аугментацией cutout размечалось улицей, а у тела стены вход тоже
пустой - сеть разучивалась видеть стены (val CE росла вчетверо, IoU стен 0,84 -> 0,75, на скане kzn план
пропал). Исправлено (cutout снова ignore); `v5_ft2` - заново с весов `v4_ft`.

**Если сессия оборвалась** - снова *Выполнить все*: продолжится с `runs/<RUN_NAME>/last.pt` (старые веса второй раз
не грузятся).

**Сравнение.** Ячейка «Счёт плана: старая модель против дообученной» прогоняет обе на одном наборе уровней val нового
датасета: видно, что дало дообучение. Остальные ячейки разбора - как в `colab_train.ipynb`.

In [ ]:
#@title Настройки { display-mode: "form" }
DRIVE_DIR = '/content/drive/MyDrive/plannet'  #@param {type:"string"}
DATASET_ZIP = 'dataset_v5.zip'  #@param {type:"string"}
RUN_NAME = 'v5_ft2'  #@param {type:"string"}
REPO_URL = 'https://github.com/BykovVadimCV/E57toARCHICAD'  #@param {type:"string"}
BRANCH = 'ccr-25c1b96e-m10uo0'  #@param {type:"string"}
#@markdown ---
EPOCHS = 30  #@param {type:"integer"}
BATCH = 8  #@param {type:"integer"}
CROP = 512  #@param {type:"integer"}
LR = 0.0003  #@param {type:"number"}
WORKERS = 2  #@param {type:"integer"}
ACCUM_STEPS = 1  #@param {type:"integer"}
USE_WEIGHTED_SAMPLING = False  #@param {type:"boolean"}
USE_FOCAL_LOSS = False  #@param {type:"boolean"}
VEC_EVERY = 5  #@param {type:"integer"}
VEC_LEVELS = 48  #@param {type:"integer"}
INIT_RUN = 'v4_ft'  #@param {type:"string"}
#@markdown ---
#@markdown Разбор: часть датасета, TTA, сравнение с TTA/без, предел уровней (0 - все), реальные E57 -> cellplan -> IFC
EVAL_SPLIT = 'test'  #@param ["test", "val"]
TTA = 'none'  #@param ["none", "flip", "d4"]
COMPARE_TTA = False  #@param {type:"boolean"}
EVAL_LIMIT = 0  #@param {type:"integer"}
REAL_TO_IFC = True  #@param {type:"boolean"}

import os, sys
IN_COLAB = 'google.colab' in sys.modules
# проверка вне Colab: PLANNET_LOCAL=<папка, как plannet на Drive>, PLANNET_REPO=<код>, PLANNET_WORK=<рабочая папка>
if not IN_COLAB and os.environ.get('PLANNET_LOCAL'):
    DRIVE_DIR = os.environ['PLANNET_LOCAL']
WORK = os.environ.get('PLANNET_WORK', '/content')
REPO = os.environ.get('PLANNET_REPO', f'{WORK}/E57toARCHICAD')
DATA = f'{WORK}/data/{os.path.splitext(DATASET_ZIP)[0]}'     # свой каталог на архив: другой архив - не старые данные
RUN = f'{DRIVE_DIR}/runs/{RUN_NAME}'
for k in ('EPOCHS', 'BATCH', 'CROP', 'WORKERS', 'EVAL_LIMIT'):   # для быстрой проверки ноутбука
    if os.environ.get('PLANNET_' + k):
        globals()[k] = int(os.environ['PLANNET_' + k])
print('Drive:', DRIVE_DIR, '| прогон:', RUN, '| датасет:', DATASET_ZIP)
print(f'батч {BATCH} x накопление {ACCUM_STEPS} = {BATCH * ACCUM_STEPS}; weighted sampling {USE_WEIGHTED_SAMPLING}; '
      f'focal {USE_FOCAL_LOSS}; разбор на {EVAL_SPLIT}, TTA {TTA}')

In [ ]:
#@title Google Drive и GPU
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory / 2**30:.0f} ГБ')
else:
    print('GPU нет - обучение на CPU будет очень медленным. Среда выполнения → Сменить тип среды → GPU.')
os.makedirs(RUN, exist_ok=True)
assert os.path.exists(f'{DRIVE_DIR}/{DATASET_ZIP}') or os.path.exists(f'{DATA}/meta.json'), \
    f'нет {DRIVE_DIR}/{DATASET_ZIP}'

In [ ]:
#@title Код (ветка репозитория)
import subprocess, shutil

def run(cmd, cwd=None, quiet=False):
    """Команда с выводом по строкам (видно ход обучения); ошибка - исключение."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet:
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'код {p.returncode}: {" ".join(cmd)}\n' + ''.join(out[-20:]))
    return ''.join(out)

def repo_url():
    """Закрытый репозиторий - токен из секретов Colab (GITHUB_TOKEN); в выводе не печатается."""
    tok = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            tok = userdata.get('GITHUB_TOKEN')
        except Exception:
            tok = None
    return REPO_URL.replace('https://', f'https://x-access-token:{tok}@') if tok else REPO_URL

if IN_COLAB:
    same = os.path.exists(f'{REPO}/.git') and \
        run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], quiet=True).strip() == BRANCH
    if not same:                                   # другой репозиторий или ветка - чистый клон
        shutil.rmtree(REPO, ignore_errors=True)
        run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, repo_url(), REPO])
    else:
        run(['git', '-C', REPO, 'pull', '-q', '--ff-only'])
sys.path.insert(0, REPO)
os.chdir(REPO)
import cv2, numpy as np
from simscan.netinput import CLASSES, CHANNELS
NAMES = ['улица', 'помещение', 'стена', 'дверь', 'окно', 'проём', 'мусор', 'пустота']
print(run(['git', '-C', REPO, 'log', '-1', '--format=код: %h %s'], quiet=True).strip())

In [ ]:
#@title Датасет: с Drive на локальный диск Colab; состав
import json, time, zipfile, collections
if not os.path.exists(f'{DATA}/meta.json'):
    t = time.time()
    tmp = f'{WORK}/data/_unzip'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    local_zip = f'{WORK}/{DATASET_ZIP}'
    if not os.path.exists(local_zip):
        print('копирую с Drive...'); shutil.copy(f'{DRIVE_DIR}/{DATASET_ZIP}', local_zip)
    with zipfile.ZipFile(local_zip) as z:
        top = z.namelist()[0].split('/')[0]
        print('распаковываю', len(z.namelist()), 'файлов...'); z.extractall(tmp)
    os.rename(f'{tmp}/{top}', DATA); shutil.rmtree(tmp)
    os.remove(local_zip)
    print(f'готово за {time.time() - t:.0f} с')
meta = json.load(open(f'{DATA}/meta.json'))
splits = {s: open(f'{DATA}/splits/{s}.txt').read().split() for s in ('train', 'val', 'test')}
print('уровней:', {k: len(v) for k, v in splits.items()}, '| пиксель', meta['pixel_m'], 'м')
assert splits['train'] and splits['val'], 'в датасете нет train или val'
if not splits['test']:
    print('ВНИМАНИЕ: test пуст - разбор будет на val (он же выбирал best.pt, числа завышены)')
    EVAL_SPLIT = 'val'

lab_ver = collections.Counter(json.load(open(f'{DATA}/scenes/{i}/scene.json')).get('labels', '?')
                              for i in splits['train'][::max(1, len(splits['train']) // 50)])
print('разметка:', dict(lab_ver))
if set(lab_ver) - {'labels-4'}:
    print('ВНИМАНИЕ: датасет собран старым кодом (не labels-4): нет открытых балконов, а пол верхнего этажа '
          'двухуровневых квартир может быть потолком нижнего; неотснятое (255) загрузчик всё равно читает как улицу')

from simscan.evaluate import level_tags
tags = collections.Counter()
scn = [json.load(open(f'{DATA}/scenes/{i}/scene.json')) for i in splits['train']]
for s in scn:
    for k, v in level_tags(s).items():
        tags[(k, v)] += 1
n_old = tags[('генератор', 'старый: без наклонов и дуг')]
if n_old:
    print(f'ВНИМАНИЕ: {n_old} из {len(scn)} уровней train собраны старым генератором (без наклонов и дуг) - '
          'пересоберите датасет кодом ветки ' + BRANCH + ', иначе сеть не увидит неортогональных стен')
print('состав train (уровней):')
for k in sorted({k for k, _ in tags}):
    print(f'  {k}: ' + ', '.join(f'{v} {n} ({n / len(scn):.0%})' for (kk, v), n in sorted(tags.items()) if kk == k))
cnt = np.zeros(256)
for i in splits['train'][:300]:
    cnt += np.bincount(cv2.imread(f'{DATA}/scenes/{i}/sem.png', cv2.IMREAD_UNCHANGED).ravel(), minlength=256)
print('доля пикселей:', {n: f'{c / cnt[:8].sum():.1%}' for n, c in zip(NAMES, cnt[:8])})

In [ ]:
#@title Как выглядят образцы: вход, разметка, направление стен
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from simscan.evaluate import _rgb as rgb, _sem_rgb as sem_rgb
from simscan.infer import COLORS
from simscan.torchdata import FloorplanSamples

def orient_rgb(o, sem):
    """Направление стены цветом (оттенок = угол по модулю 180°): наклонные и дуги видны сразу."""
    import matplotlib.colors as mc
    h = (np.arctan2(o[1], o[0]) / (2 * np.pi)) % 1.0
    m = np.isin(sem, (2, 3, 4, 5))
    im = mc.hsv_to_rgb(np.stack([h, np.ones_like(h), m.astype(float)], -1))
    im[~m] = 1.0
    return im

ds = FloorplanSamples(DATA, 'train', crop=CROP, train=True, seed=1, weighted_sampling=False)
fig, ax = plt.subplots(3, 4, figsize=(18, 13))
for k, idx in enumerate(np.random.default_rng(0).permutation(len(ds))[:4]):
    s = ds[int(idx)]
    ax[0, k].imshow(rgb(s['input'])); ax[0, k].set_title(s['id'], fontsize=8)
    ax[1, k].imshow(sem_rgb(s['sem']))
    ax[2, k].imshow(orient_rgb(s['orient'], s['sem']))
for a in ax.ravel(): a.axis('off')
ax[2, 0].set_title('направление стен (цвет - угол)', fontsize=9, loc='left')
fig.legend([Patch(color=COLORS[i] / 255) for i in range(8)], NAMES, loc='lower center', ncol=8)
plt.show()

In [ ]:
#@title Обучение: старт с весов INIT_RUN, после обрыва - с last.pt
cmd = [sys.executable, '-m', 'simscan', 'train', '--data', DATA, '--out', RUN, '--epochs', str(EPOCHS),
       '--batch', str(BATCH), '--crop', str(CROP), '--lr', str(LR), '--workers', str(WORKERS), '--resume']
if ACCUM_STEPS > 1:
    cmd += ['--accum-steps', str(ACCUM_STEPS)]
if USE_WEIGHTED_SAMPLING:
    cmd += ['--weighted-sampling']
if USE_FOCAL_LOSS:
    cmd += ['--use-focal']
cmd += ['--vec-every', str(VEC_EVERY), '--vec-levels', str(VEC_LEVELS)]
if not os.path.exists(f'{RUN}/last.pt'):                         # старые веса - только при старте прогона
    src = f'{DRIVE_DIR}/runs/{INIT_RUN}'
    init = next((f'{src}/{f}' for f in ('best.pt', 'best_pixel.pt', 'last.pt') if os.path.exists(f'{src}/{f}')), None)
    assert init, f'в {src} нет best.pt, best_pixel.pt или last.pt'
    print('старт с весов', init)
    cmd += ['--init', init]
else:
    print('продолжаю прогон', RUN, 'с last.pt')
if os.environ.get('PLANNET_MAX_STEPS'):
    cmd += ['--max-steps', os.environ['PLANNET_MAX_STEPS']]
print('Команда:', ' '.join(cmd))
run(cmd)

In [ ]:
#@title Кривые обучения и диагноз
h = [json.loads(l) for l in open(f'{RUN}/history.jsonl')]
h = list({r['epoch']: r for r in h}.values())             # повтор эпохи после обрыва - последняя запись
ep = [r['epoch'] for r in h]
parts = [k for k in ('ce', 'dice', 'cldice', 'dist', 'orient', 'heights') if k in h[0]]
fig, ax = plt.subplots(2, 3, figsize=(19, 9))
ax[0, 0].plot(ep, [r['total'] for r in h], label='train'); ax[0, 0].plot(ep, [r.get('val_total') for r in h], label='val')
ax[0, 0].set_title('потеря (сумма)'); ax[0, 0].legend()
for k in parts:
    l, = ax[0, 1].plot(ep, [r[k] for r in h], label=k)
    ax[0, 1].plot(ep, [r.get('val_' + k) for r in h], '--', color=l.get_color())
ax[0, 1].set_yscale('log'); ax[0, 1].set_title('части потери: сплошная - train, пунктир - val'); ax[0, 1].legend(fontsize=8)
ax[0, 2].plot(ep, [r['lr'] for r in h]); ax[0, 2].set_title('шаг обучения (LR)')
mi = [r.get('miou_walls') for r in h]
bi = int(np.nanargmax(np.array(mi, float)))
ax[1, 0].plot(ep, mi); ax[1, 0].axvline(ep[bi], color='g', ls=':'); ax[1, 0].set_title(f'val mIoU стены и проёмов (лучшая эпоха {ep[bi]})')
iou = np.array([[np.nan if v is None else v for v in r.get('iou', [None] * 8)] for r in h], float)
for c in range(8):
    ax[1, 1].plot(ep, iou[:, c], label=NAMES[c])
ax[1, 1].set_title('val IoU по классам'); ax[1, 1].legend(fontsize=7)
gap = [r.get('val_total', np.nan) - r['total'] for r in h]
ax[1, 2].plot(ep, gap); ax[1, 2].axhline(0, color='k', lw=0.5); ax[1, 2].set_title('val - train (рост = переобучение)')
plt.tight_layout(); plt.show()

vt = np.array([r.get('val_total', np.nan) for r in h], float)
best = h[bi]
print(f"лучшая эпоха {best['epoch']} из {ep[-1]}: mIoU стены и проёмов {best['miou_walls']:.3f}")
if len(h) >= 5:
    tail = np.array(mi[-5:], float)
    print('- последние 5 эпох: mIoU', np.round(tail, 3), '->', 'ещё растёт: больше эпох поможет' if tail[-1] > tail[0] + 0.005
          else 'вышел на плато')
    if np.nanargmin(vt) < len(vt) - 5 and vt[-1] > np.nanmin(vt) * 1.05:
        print(f'- val-потеря минимальна на эпохе {ep[int(np.nanargmin(vt))]} и выросла на {vt[-1] / np.nanmin(vt) - 1:.0%}: '
              'переобучение - больше сцен/вариантов или меньше эпох')
    worst_c = min((c for c in (2, 3, 4, 5) if not np.isnan(iou[bi, c])), key=lambda c: iou[bi, c])
    print(f'- слабейший класс конструкции на лучшей эпохе: {NAMES[worst_c]} (IoU {iou[bi, worst_c]:.3f})')

# план (best.pt выбирается по нему): счёт и составляющие на уровнях val
hv = [r for r in h if r.get('vec_geo_score') is not None]
if hv:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    e = [r['epoch'] for r in hv]
    ax[0].plot(e, [r['vec_geo_score'] for r in hv], 'o-'); ax[0].set_title('счёт плана (стены 0,5, двери 0,3, окна 0,2)')
    for k, n in (('vec_axis_f1', 'оси стен F1'), ('vec_door_f1', 'двери F1'), ('vec_window_f1', 'окна F1')):
        ax[1].plot(e, [r.get(k) for r in hv], 'o-', label=n)
    ax[1].set_title('составляющие'); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    bg = max(hv, key=lambda r: r['vec_geo_score'])
    print(f"best.pt - эпоха {bg['epoch']}: счёт плана {bg['vec_geo_score']:.3f}, ось p90 {bg.get('vec_axis_err_cm_p90')} см, "
          f"кусков на стену {bg.get('vec_pieces_per_gt_wall')}")

In [ ]:
#@title Счёт плана: старая модель против дообученной (одни и те же уровни val нового датасета)
from simscan.infer import load_model
from simscan.train import vector_eval
src = f'{DRIVE_DIR}/runs/{INIT_RUN}'
old = next((f'{src}/{f}' for f in ('best.pt', 'best_pixel.pt', 'last.pt') if os.path.exists(f'{src}/{f}')), None)
val = splits['val']
ids = [val[k] for k in np.linspace(0, len(val) - 1, min(VEC_LEVELS, len(val))).astype(int)]
print(f'{"модель":28s} {"план":>6s} {"оси F1":>7s} {"p90 оси, см":>12s} {"двери F1":>9s} {"окна F1":>8s}')
for name, ck in ((f'{INIT_RUN} ({os.path.basename(old)})', old), (f'{RUN_NAME} (best.pt)', f'{RUN}/best.pt')):
    m, dev = load_model(ck, log=lambda *a: None)
    vm = vector_eval(m, dev, DATA, ids, tile=CROP)
    f = lambda k: '-' if vm.get(k) is None else f'{vm[k]:.3f}'
    print(f'{name:28s} {f("geo_score"):>6s} {f("axis_f1"):>7s} {f("axis_err_cm_p90"):>12s} {f("door_f1"):>9s} {f("window_f1"):>8s}')
    del m


In [ ]:
#@title Разбор best.pt на test: классы, контур, проёмы, ориентация, группы, худшие уровни
from IPython.display import Image, display
from simscan.evaluate import evaluate, compare
lim = EVAL_LIMIT or None
rep_dir = f'{RUN}/report_{EVAL_SPLIT}' + ('' if TTA == 'none' else f'_tta_{TTA}')
t = time.time()
rep = evaluate(f'{RUN}/best.pt', DATA, EVAL_SPLIT, rep_dir, tta=TTA, tile=CROP, limit=lim)
print(f"{rep['levels']} уровней {EVAL_SPLIT} за {time.time() - t:.0f} с; отчёт: {rep_dir}")
for f in rep['figures']:
    display(Image(f, width=1100))

In [ ]:
#@title Числа и выводы
def row(name, v, note=''):
    print(f'  {name:42s} {"-" if v is None else v!s:>8s}  {note}')

print('КЛАССЫ (IoU / точность / полнота):')
for n, c in rep['classes'].items():
    print(f"  {n:10s} IoU {c['iou']}  точн. {c['precision']}  полн. {c['recall']}  доля {c['gt_share']}")
print('\nКОНТУР И ПРОЁМЫ:')
row('mIoU стены и проёмов (как best.pt)', rep['miou_walls'])
row('IoU конструкции (стена+проёмы как одно)', rep['struct_iou'])
row(f"F1 конструкции с допуском ±{rep['struct_tol']['tol_cm']} см", rep['struct_tol']['f1'], 'положение без придирки к толщине')
row('полнота скелета стен', rep['skeleton']['recall'], 'доля осей эталона, которые сеть провела')
row('точность скелета стен', rep['skeleton']['precision'], 'доля осей сети, которые есть в эталоне')
row('кусков конструкции: эталон / сеть', f"{rep['skeleton']['pieces_gt']}/{rep['skeleton']['pieces_pred']}",
    'больше у сети - рваные стены или мусорные пятна')
for n, o in rep['openings'].items():
    row(f'{n}: найдено / всего, лишних', f"{o['found']}/{o['gt']}, {o['false']}", f"F1 {o['f1']}; пропуски: {o['missed_as']}")
we = rep['opening_width_err_cm']
row('ширина найденных проёмов: MAE / смещение, см', f"{we['mae']}/{we['bias']}")
print('\nГОЛОВЫ:')
row('расстояние до грани, MAE см', rep.get('dist_mae_cm'))
for b, o in rep.get('orientation_err_deg', {}).items():
    row(f'ориентация, {b}: медиана / p90, °', f"{o['median']}/{o['p90']}", f"{o['px']} px, >5°: {o['share_over_5']}")
hm = rep.get('heights_mae_cm', {})
row('подоконник окна / верх проёма, MAE см', f"{hm.get('sill_window')}/{hm.get('head')}")
row('калибровка ECE', rep.get('calibration', {}).get('ece'), '< 0.03 - вероятностям можно верить (их берёт cellplan)')
vec = rep.get('vector')
if vec:
    print('\nВЕКТОРНЫЙ ПЛАН (vectorize по выходу сети против vector.json):')
    row('оси стен: полнота / точность (±6 см)', f"{vec['axis_recall']}/{vec['axis_precision']}")
    row('ошибка направления / толщины', f"{vec['angle_err_deg_median']}° / {vec['thickness_err_cm_median']} см")
    row('кусков на прямую стену эталона', vec['pieces_per_gt_wall'], '1 - стена одной прямой')
    row('разброс осей кусков одной стены, см (p90)', vec['axis_offset_cm_p90'], '0 - все куски на одной прямой')
    row('висящих концов на стену эталона', vec['dangling_ends_per_gt_wall'], '0 - контур замкнут')
    row('одиночных кусков (шум) всего', vec['standalone_walls_total'])
    row('проёмы: найдено / всего / предсказано', f"{vec['openings_found_total']}/{vec['openings_gt_total']}/{vec['openings_pred_total']}")
    row('помещения: IoU, предсказано / всего', f"{vec['room_iou_mean']}, {vec['rooms_pred_total']}/{vec['rooms_gt_total']}")

print('\nВЫВОДЫ:')
notes = []
sk = rep['skeleton']
if sk['recall'] is not None and sk['recall'] < 0.9:
    notes.append(f"стены рвутся: сеть проводит {sk['recall']:.0%} осей эталона - в cellplan разрыв закроет продление, "
                 "но не всегда; смотрите синие участки на карте ошибок худших уровней")
if rep['struct_iou'] is not None and rep['struct_iou'] < 0.5:
    notes.append(f"IoU конструкции {rep['struct_iou']} - сеть пока слабая: cellplan с её вероятностями может дать план хуже, "
                 "чем одни правила (сравнение - в ячейке реальных объектов)")
for n, o in rep['openings'].items():
    if not o['gt']:
        continue
    miss = max(o['missed_as'], key=o['missed_as'].get) if o['gt'] - o['found'] else None
    if o['recall'] < 0.8 and miss:
        notes.append(f"{n}: найдено {o['recall']:.0%}; чаще всего вместо него - «{miss}»")
    if o['precision'] is not None and o['precision'] < 0.8:
        notes.append(f"{n}: {o['false']} ложных - сеть ставит «{n}» там, где его нет")
oe = rep.get('orientation_err_deg', {})
a, b2 = oe.get('по осям', {}), oe.get('наклон 1-6°', {})
if a.get('median') is not None and a['median'] > 10:
    notes.append(f"ориентация не выучена (медиана {a['median']}° даже по осям) - голова orient бесполезна, cellplan её не берёт, "
                 "но для обучения это признак малого числа эпох")
elif a.get('median') is not None and b2.get('median') is not None and b2['px'] > 1000:
    if b2['median'] > 2 * max(a['median'], 0.5):
        notes.append(f"наклонные стены: ошибка направления {b2['median']}° против {a['median']}° по осям - сеть тянет "
                     "их к осям; больше сцен с наклонами (realism.p_angle_jitter) или дольше учить")
    else:
        notes.append(f"наклонные стены: направление {b2['median']}° (по осям {a['median']}°) - наклоны сеть различает")
for g, vals in rep['groups'].items():
    v = {k: x['struct_iou'] for k, x in vals.items() if x['struct_iou'] is not None and x['levels'] >= 3}
    if len(v) >= 2 and max(v.values()) - min(v.values()) > 0.05:
        lo = min(v, key=v.get)
        notes.append(f"«{g}»: хуже всего «{lo}» (IoU {v[lo]:.3f} против {max(v.values()):.3f}) - "
                     f"{vals[lo]['levels']} уровней; кандидат на долю в датасете")
if vec and vec.get('pieces_per_gt_wall') and vec['pieces_per_gt_wall'] > 1.3:
    notes.append(f"стены дробятся ({vec['pieces_per_gt_wall']} куска на стену): сеть рвёт стены - смотрите полноту скелета "
                 "и худшие уровни; векторизатор закрывает разрывы до 0,5-0,8 м")
if vec and vec.get('dangling_ends_per_gt_wall') and vec['dangling_ends_per_gt_wall'] > 0.3:
    notes.append(f"висящих концов {vec['dangling_ends_per_gt_wall']} на стену - контур не замкнут (пропуски стен у стыков)")
ece = rep.get('calibration', {}).get('ece')
if ece is not None and ece > 0.05:
    notes.append(f'вероятности переуверенные (ECE {ece}) - для cellplan уменьшить --net-w или калибровать (temperature)')
for n in notes or ['явных проблем по порогам нет - смотрите худшие уровни']:
    print(' -', n)
if vec:
    from simscan.evaluate import geo_score
    row('двери: F1 / ошибка ширины, см', f"{vec.get('door_f1')}/{vec.get('door_width_err_cm')}")
    row('окна: F1 / ошибка ширины, см', f"{vec.get('window_f1')}/{vec.get('window_width_err_cm')}")
    row('СЧЁТ ПЛАНА (как выбирается best.pt)', geo_score(vec), 'стены 0,5, двери 0,3, окна 0,2')

In [ ]:
#@title Сравнение: без TTA и с TTA (если COMPARE_TTA; с TTA = выбранный режим, иначе flip)
if COMPARE_TTA:
    alt = 'none' if TTA != 'none' else 'flip'
    other = evaluate(f'{RUN}/best.pt', DATA, EVAL_SPLIT, f'{RUN}/report_{EVAL_SPLIT}' + ('' if alt == 'none' else f'_tta_{alt}'),
                     tta=alt, tile=CROP, limit=lim)
    a, b = (other, rep) if TTA != 'none' else (rep, other)
    print(f"{'метрика':40s} {'без TTA':>9s} {'с TTA':>9s} {'разница':>9s}")
    for k, x, y, d in compare(a, b):
        print(f"{k:40s} {x!s:>9s} {y!s:>9s} {('+' if d and d > 0 else '') + str(d):>9s}")
else:
    print('COMPARE_TTA выключен')

In [ ]:
#@title Реальные объекты: E57 из plannet/real -> сеть -> cellplan -> IFC
import glob, collections
from IPython.display import Image, display   # ячейка разбора могла не запускаться
real = sorted(glob.glob(f'{DRIVE_DIR}/real/*.e57'))
if not real:
    print(f'нет файлов {DRIVE_DIR}/real/*.e57 - положите реальные сканы, чтобы посмотреть сеть на них')
else:
    cmd = [sys.executable, '-m', 'simscan', 'infer', f'{RUN}/best.pt', *real, '--out', f'{RUN}/pred', '--tile', str(CROP),
           '--tta', TTA, '--plan']
    run(cmd)
    for p in sorted(glob.glob(f'{RUN}/pred/*/L*/pred.png')):        # классы и векторный план только по сети
        print(p); display(Image(p, width=900))
        q = os.path.join(os.path.dirname(p), 'plan.png')
        if os.path.exists(q):
            display(Image(q, width=700))
    if REAL_TO_IFC:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'ifcopenshell'], quiet=True)

        def summary(out):
            """Короткая сводка elements.json cellplan по уровням."""
            el = json.load(open(f'{out}/elements.json'))
            res = []
            for lv in el['levels']:
                kinds = collections.Counter(o['opening_type'] for o in lv['openings'])
                res.append(f"помещений {len(lv['rooms'])} ({sum(r['area'] for r in lv['rooms']):.1f} м²), "
                           f"стен {len(lv['walls'])}, проёмов {dict(kinds)}, "
                           f"сверка стен: полнота {lv['check']['wall_recall']:.0%}, точность {lv['check']['wall_precision']:.0%}")
            return res

        for e57 in real:
            stem = os.path.splitext(os.path.basename(e57))[0]
            out, out_rules = f'{RUN}/plan/{stem}', f'{RUN}/plan/{stem}_rules'
            run([sys.executable, '-m', 'simscan', 'cellplan', e57, '--pred', f'{RUN}/pred/{stem}', '--out', out], quiet=True)
            run([sys.executable, '-m', 'simscan', 'cellplan', e57, '--out', out_rules], quiet=True)
            print(f'\n{stem}: план с сетью против одних правил')
            for k, (a, b) in enumerate(zip(summary(out), summary(out_rules)), 1):
                print(f'  L{k} с сетью:  {a}\n  L{k} правила: {b}')
            plans = sorted(glob.glob(f'{out}/plan_L*.json'))
            run([sys.executable, '-m', 'simscan', 'export-ifc', *plans, '--out', f'{out}/{stem}.ifc'], quiet=True)
            for p in sorted(glob.glob(f'{out}/plan*.png')) + sorted(glob.glob(f'{out_rules}/plan*.png')):
                print(p); display(Image(p, width=900))
            print('IFC (с сетью):', f'{out}/{stem}.ifc')

**Где результат.** На Drive в `plannet/runs/<имя>/`: `best.pt` - веса; `report_test/` - `report.json` (все числа),
`levels.csv` (строка на уровень - удобно сортировать худшие), картинки разбора; `pred/` и `plan/` - реальные объекты
(классы, вероятности, план `cellplan`, IFC). Применить локально:
`python -m simscan infer best.pt объект.e57 --out pred` → `python -m simscan cellplan объект.e57 --pred pred/объект --out plan`
→ `python -m simscan export-ifc plan/plan_L*.json --out объект.ifc`.